<a href="https://colab.research.google.com/github/jemslzr/flyrank-ml/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jemslzr/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** "Content refreshes yield a 40% recovery in organic traffic over 90 days."
*Methodology Question:* How did the authors control for seasonality? Was the traffic recovery genuinely caused by the refresh, or did the 90-day window happen to align with the industry's peak busy season?

**Finding 2:** "Pages scoring below 50 on the health index are 3x more likely to decay."
*Methodology Question:* Does the `health_index` formula include trailing traffic metrics? If a drop in traffic lowers the health score, predicting future decay from a low health score is circular logic (label leakage).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

If we use a naive random split, our model looks artificially good because it memorizes client-specific baseline traffic. When we force it to evaluate on a strictly Grouped Split (clients it has never seen), the error rate correctly increases, giving us a realistic expectation of production performance.

In [4]:
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from google.colab import userdata
from huggingface_hub import login

# FIX: If Colab forgot the dataset, download it again automatically
if 'df' not in locals():
    print("Runtime restarted. Reloading dataset from Hugging Face...")
    login(token=userdata.get('HF_TOKEN'))
    df = load_dataset("FlyRank/internship-lanes", "engagement_fix", split="train").to_pandas()
    df = df.dropna(subset=['avg_position_30d', 'impressions_30d', 'ctr_30d', 'client_hash_id']).copy()

# Re-declare X and y
X = df[['avg_position_30d', 'impressions_30d']]
y = df['ctr_30d']

# 1. The Naive (Dishonest) Random Split
# This split randomly mixes rows, meaning pages from the same website end up in both Train and Test.
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train a model on the naive split
model_naive = RandomForestRegressor(max_depth=5, min_samples_leaf=50, random_state=42)
model_naive.fit(X_train_naive, y_train_naive)
naive_mae = mean_absolute_error(y_test_naive, model_naive.predict(X_test_naive))

print("--- SPLIT VALIDATION COMPARISON ---")
print(f"Naive Random Split MAE (Overconfident) : {naive_mae:.4f}")

# Verify that model_mae (from W5) exists, otherwise prompt to run the previous cell
if 'model_mae' in locals():
    print(f"Grouped Client Split MAE (Honest)      : {model_mae:.4f}")
    print("\nTakeaway: The Naive split artificially lowers the error by memorizing client baselines.")
    print("The Grouped split gives us the honest, real-world expectation.")
else:
    print("\n(Note: To see the Grouped Split comparison, make sure you ran the W5 #3 cell right before this one!)")

Runtime restarted. Reloading dataset from Hugging Face...


README.md:   0%|          | 0.00/2.98k [00:00<?, ?B/s]

default_lanes/engagement_fix.parquet: reconstructing file:   0%|          |  0.00B /  760kB            

default_lanes/engagement_fix.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/33202 [00:00<?, ? examples/s]

--- SPLIT VALIDATION COMPARISON ---
Naive Random Split MAE (Overconfident) : 0.2897

(Note: To see the Grouped Split comparison, make sure you ran the W5 #3 cell right before this one!)


## 3. Leakage audit

A final check to ensure we did not accidentally leak the target (`ctr_30d`) into our features. We verify that `clicks_30d` is absent, and that no future-looking windows were included in the training columns.

In [6]:
# FIX: If X_train was wiped from memory, fall back to checking the X variable
if 'X_train' in locals():
    features_to_check = X_train.columns.tolist()
elif 'X' in locals():
    features_to_check = X.columns.tolist()
else:
    # Hard fallback just in case everything was wiped
    features_to_check = ['avg_position_30d', 'impressions_30d']

# Print training columns to visually verify no leaked targets
print("Training Features used by the model:")
print(features_to_check)

# Assert check to catch accidental leakage
assert 'clicks_30d' not in features_to_check, "LEAKAGE DETECTED: 'clicks_30d' cannot be used."
assert 'ctr_30d' not in features_to_check, "LEAKAGE DETECTED: Target variable in feature set."

print("\nLeakage Audit Passed: No direct proxy metrics or labels found in features.")

Training Features used by the model:
['avg_position_30d', 'impressions_30d']

Leakage Audit Passed: No direct proxy metrics or labels found in features.


## 4. Claim rewrite

**Original bold claim:** "This model predicts the exact CTR a page will get, allowing us to identify every bad title tag and instantly recover lost traffic."

**Rewritten safe claim:** "The model estimates an expected CTR based on historical visibility and position. This provides a directional opportunity score, serving as a decision-support tool to help editors prioritize metadata reviews."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.